# Fine-tune DINOv2 on FGVC-Aircraft (+ optional Wikimedia extras)

**Aviation Intelligence System — ZHAW FS26 Semester Project**

This notebook trains a stronger 100-class aircraft classifier than the original ViT baseline:

* **Backbone:** `facebook/dinov2-base` — self-supervised features that transfer well to fine-grained tasks
* **Epochs:** 20 (cosine schedule with warmup)
* **Augmentation:** RandAugment + RandomErasing + Mixup
* **Optional extra training data:** Wikimedia Commons photos scraped with `src/cv/scrape_extra_images.py` and uploaded as `data/raw/extra_images/<variant>/*.jpg`

Run on a free Colab T4 (~60 min). Set runtime → T4 GPU before starting.

In [ ]:
!pip install -q transformers==4.46.0 datasets==3.0.0 accelerate==1.0.0 torchvision==0.20.1 evaluate==0.4.3 huggingface_hub timm

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # paste a WRITE token from https://huggingface.co/settings/tokens

## 1. Download FGVC-Aircraft

In [ ]:
from torchvision.datasets import FGVCAircraft
from pathlib import Path
DATA = Path('/content/data')
DATA.mkdir(exist_ok=True)
for split in ('train', 'val', 'test'):
    FGVCAircraft(root=str(DATA), split=split, annotation_level='variant', download=True)
print('FGVC ready.')

## 2. (Optional) Add scraped Wikimedia Commons images

Run `PYTHONPATH=. uv run python -m src.cv.scrape_extra_images` locally first, then **zip** `data/raw/extra_images/` and upload it via the Colab files panel as `extra_images.zip`. The next cell unpacks it and merges it into the training set.

Skip this section if you only want to train on FGVC.

In [ ]:
import os, zipfile
EXTRA_DIR = Path('/content/extra_images')
if Path('extra_images.zip').exists():
    with zipfile.ZipFile('extra_images.zip') as z:
        z.extractall('/content/')
    print('Extras extracted to', EXTRA_DIR)
else:
    print('No extra_images.zip found — training on FGVC only.')

## 3. Merged dataset wrapper

Builds a `(file_path, class_index)` list that combines FGVC train + any extras whose folder name matches an FGVC class.

In [ ]:
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from transformers import AutoImageProcessor

MODEL_ID = 'facebook/dinov2-base'
processor = AutoImageProcessor.from_pretrained(MODEL_ID)

fgvc_train = FGVCAircraft(root=str(DATA), split='train', annotation_level='variant', download=False)
fgvc_val   = FGVCAircraft(root=str(DATA), split='val', annotation_level='variant', download=False)
fgvc_test  = FGVCAircraft(root=str(DATA), split='test', annotation_level='variant', download=False)
classes = fgvc_train.classes
class_to_idx = {c: i for i, c in enumerate(classes)}

# canonicalize folder names: 'A380' -> 'A380', '777-200' -> '777-200', 'Cessna 172' -> 'Cessna_172' / 'Cessna 172'
def folder_to_class(name):
    name = name.replace('_', ' ').strip()
    if name in class_to_idx: return class_to_idx[name]
    name2 = name.replace(' ', '-')
    if name2 in class_to_idx: return class_to_idx[name2]
    return None

extra_pairs = []
if EXTRA_DIR.exists():
    for folder in EXTRA_DIR.iterdir():
        if not folder.is_dir(): continue
        cls = folder_to_class(folder.name)
        if cls is None:
            print(f'  ! no class match for {folder.name}'); continue
        for f in sorted(folder.glob('*.jpg')):
            extra_pairs.append((str(f), cls))
    print(f'Extras: {len(extra_pairs)} images merged')

class MergedDataset(Dataset):
    def __init__(self, fgvc, extras, train: bool):
        self.fgvc = fgvc
        self.extras = extras
        if train:
            self.tx = transforms.Compose([
                transforms.Resize((256, 256)),
                transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.RandAugment(num_ops=2, magnitude=9),
                transforms.ToTensor(),
                transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
                transforms.RandomErasing(p=0.25),
            ])
        else:
            self.tx = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
            ])
    def __len__(self): return len(self.fgvc) + len(self.extras)
    def __getitem__(self, i):
        if i < len(self.fgvc):
            img, lbl = self.fgvc[i]
        else:
            f, lbl = self.extras[i - len(self.fgvc)]
            img = Image.open(f).convert('RGB')
        return {'pixel_values': self.tx(img.convert('RGB')), 'labels': int(lbl)}

train_ds = MergedDataset(fgvc_train, extra_pairs, train=True)
val_ds   = MergedDataset(fgvc_val, [], train=False)
test_ds  = MergedDataset(fgvc_test, [], train=False)
print(f'classes={len(classes)}  train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}')

## 4. Train

In [ ]:
import numpy as np
from transformers import AutoModelForImageClassification, Trainer, TrainingArguments

model = AutoModelForImageClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(classes),
    id2label={i: c for i, c in enumerate(classes)},
    label2id={c: i for i, c in enumerate(classes)},
    ignore_mismatched_sizes=True,
)

def metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, -1)
    top5 = np.argsort(logits, -1)[:, -5:]
    return {
        'accuracy': float((preds == labels).mean()),
        'top5_accuracy': float(np.mean([l in t for l, t in zip(labels, top5)])),
    }

args = TrainingArguments(
    output_dir='/content/aircraft-vit',
    num_train_epochs=20,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=5e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    weight_decay=0.05,
    label_smoothing_factor=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_steps=50,
    fp16=True,
    report_to='none',
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=metrics)
trainer.train()

## 5. Evaluate on the held-out test split

In [ ]:
test_metrics = trainer.evaluate(test_ds)
print('TEST:', test_metrics)

In [ ]:
import torch
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt, seaborn as sns
from torch.utils.data import DataLoader
preds_all, labels_all = [], []
model.eval().cuda()
dl = DataLoader(test_ds, batch_size=64, num_workers=2)
with torch.no_grad():
    for batch in dl:
        out = model(pixel_values=batch['pixel_values'].cuda()).logits.argmax(-1).cpu().numpy()
        preds_all.extend(out); labels_all.extend(batch['labels'].numpy())
print(classification_report(labels_all, preds_all, target_names=classes, zero_division=0))
cm = confusion_matrix(labels_all, preds_all)
plt.figure(figsize=(14, 12)); sns.heatmap(cm, cmap='magma'); plt.title('Confusion matrix — DINOv2 (test split)'); plt.show()

## 6. Push to Hugging Face Hub

In [ ]:
REPO = 'dubattim/aviation-intelligence-vit-fgvc'
trainer.save_model('/content/aircraft-vit')
processor.save_pretrained('/content/aircraft-vit')
model.push_to_hub(REPO)
processor.push_to_hub(REPO)
print(f'Done → https://huggingface.co/{REPO}')

The deployed Space pulls from this exact repo at runtime — no further code change needed once the push completes.